# 03 - PAN-12 and Coded Chat Overlap Check

This notebook checks for exact-text overlap between the 56 coded chats and PAN-12 before coded-chat trained predictions are used as external features. Exact normalized hashes may miss paraphrased duplicates, but they provide a reproducible lower-bound leakage check.

No overlap cases are removed automatically. Matches are reported for human review in project documentation.

In [ ]:
from pathlib import Path
import hashlib
import re
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
REPORT_DIR = WORKSPACE / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PAN12 = WORKSPACE / "pan12-sexual-predator-identification-training-corpus-2012-05-01.xml"
TEST_PAN12 = WORKSPACE / "pan12-sexual-predator-identification-test-corpus-2012-05-17.xml"
CODED_MESSAGES = PROCESSED_DIR / "coded_stage_messages.csv"

for required in [TRAIN_PAN12, TEST_PAN12, CODED_MESSAGES]:
    if not required.exists():
        raise FileNotFoundError(f"Required input not found: {required}")


## Parsing policy

PAN-12 XML is parsed directly for conversation ID, message line, author, time, and text. The coded-chat file contains the retained predator-coded rows and source chat IDs already parsed by notebook 01.

In [ ]:
DOCTYPE_PATTERN = re.compile(r"<!DOCTYPE[^>]*>", flags=re.IGNORECASE | re.DOTALL)
BARE_AMPERSAND_PATTERN = re.compile(r"&(?!#\d+;|#x[0-9A-Fa-f]+;|[A-Za-z_][A-Za-z0-9_.-]*;)")
WHITESPACE_PATTERN = re.compile(r"\s+")


def parse_xml(path: Path) -> ET.Element:
    raw = path.read_text(encoding="utf-8-sig")
    raw = DOCTYPE_PATTERN.sub("", raw)
    raw = BARE_AMPERSAND_PATTERN.sub("&amp;", raw)
    return ET.fromstring(raw)


def normalize_text(text: object) -> str:
    if text is None:
        return ""
    normalized = str(text).lower()
    normalized = WHITESPACE_PATTERN.sub(" ", normalized)
    normalized = normalized.strip()
    return normalized


def text_hash(text: object) -> str:
    normalized = normalize_text(text)
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest() if normalized else ""


def load_pan12(path: Path, split_name: str) -> pd.DataFrame:
    root = parse_xml(path)
    rows = []
    for conversation in root.findall(".//conversation"):
        conversation_id = conversation.attrib.get("id", "")
        for message in conversation.findall("./message"):
            line = message.attrib.get("line")
            author_node = message.find("./author")
            time_node = message.find("./time")
            text_node = message.find("./text")
            text = "" if text_node is None or text_node.text is None else text_node.text.strip()
            rows.append({
                "split": split_name,
                "conversation_id": conversation_id,
                "line": int(line) if line and line.isdigit() else np.nan,
                "author": "" if author_node is None or author_node.text is None else author_node.text.strip(),
                "time": "" if time_node is None or time_node.text is None else time_node.text.strip(),
                "text": text,
                "text_hash": text_hash(text),
            })
    return pd.DataFrame(rows)


pan12_train = load_pan12(TRAIN_PAN12, "train")
pan12_test = load_pan12(TEST_PAN12, "test")
pan12 = pd.concat([pan12_train, pan12_test], ignore_index=True)

print("PAN-12 train messages:", len(pan12_train))
print("PAN-12 test messages:", len(pan12_test))


## Exact overlap results

Normalization is limited to lowercasing and whitespace collapsing. Punctuation and spelling are intentionally preserved so that a match is evidence of a strong duplicate rather than broad topical similarity.

In [ ]:
coded = pd.read_csv(CODED_MESSAGES)
coded["text"] = coded["text"].fillna("")
coded["text_hash"] = coded["text"].map(text_hash)

coded_hashes = set(coded.loc[coded["text_hash"] != "", "text_hash"])
pan12_matches = pan12[pan12["text_hash"].isin(coded_hashes)].copy()

coded_match_index = coded.set_index("text_hash")
match_rows = []
for _, pan_row in pan12_matches.iterrows():
    same_hash = coded[coded["text_hash"] == pan_row["text_hash"]]
    for _, coded_row in same_hash.iterrows():
        match_rows.append({
            "split": pan_row["split"],
            "pan12_conversation_id": pan_row["conversation_id"],
            "pan12_line": pan_row["line"],
            "pan12_author": pan_row["author"],
            "coded_chat_id": coded_row["chat_id"],
            "coded_source_idx": coded_row["msg_idx"],
            "coded_category": coded_row["category"],
            "normalized_text_hash": pan_row["text_hash"],
        })

matches = pd.DataFrame(match_rows, columns=[
    "split", "pan12_conversation_id", "pan12_line", "pan12_author",
    "coded_chat_id", "coded_source_idx", "coded_category", "normalized_text_hash"
])

summary = pd.DataFrame([
    {"metric": "coded_unique_nonempty_text_hashes", "value": len(coded_hashes)},
    {"metric": "pan12_unique_matching_text_hashes", "value": matches["normalized_text_hash"].nunique() if not matches.empty else 0},
    {"metric": "pan12_matching_message_rows", "value": len(matches)},
    {"metric": "matched_pan12_conversations", "value": matches["pan12_conversation_id"].nunique() if not matches.empty else 0},
])

display(summary)
display(matches.head(50))


## Save evidence for the next notebooks

The detailed match table should be reviewed before excluding any conversations. If overlap is found, this notebook documents it without silently altering datasets.

In [ ]:
summary.to_csv(REPORT_DIR / "overlap_summary.csv", index=False)
matches.to_csv(REPORT_DIR / "exact_overlap_matches.csv", index=False)

print("Saved overlap summary to:", REPORT_DIR / "overlap_summary.csv")
print("Saved exact overlap matches to:", REPORT_DIR / "exact_overlap_matches.csv")
